<a href="https://colab.research.google.com/github/RamyaSri2222/Data-Science/blob/main/ecommer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded=files.upload()

Saving online_shoppers_intention.csv to online_shoppers_intention (1).csv


**REQUIRED LIBRARIE IMPORT** **SUCCESSFULLY**

In [ ]:
import time
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as imbpipeline
from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

df = pd.read_csv("online_shoppers_intention.csv")

df['Weekend'] = df['Weekend'].replace((True, False), (1, 0))
df['Revenue'] = df['Revenue'].replace((True, False), (1, 0))

condition = df['VisitorType'] == 'Returning_Visitor'
df['Returning_Visitor'] = np.where(condition, 1, 0)
df = df.drop(columns=['VisitorType'])

ordinal_encoder = OrdinalEncoder()
df['Month'] = ordinal_encoder.fit_transform(df[['Month']])

result = df[df.columns[1:]].corr()['Revenue']
result1 = result.sort_values(ascending=False)

X = df.drop(['Revenue'], axis=1)
y = df['Revenue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

def model_pipeline(X, model):

    n_c = X.select_dtypes(exclude=['object']).columns.tolist()
    c_c = X.select_dtypes(include=['object']).columns.tolist()

    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant')),
        ('scaler', MinMaxScaler())
    ])

    categorical_pipeline = Pipeline([
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer([
        ('numeric', numeric_pipeline, n_c),
        ('categorical', categorical_pipeline, c_c)
    ], remainder='passthrough')

    final_steps = [
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=1)),
        ('feature_selection', SelectKBest(score_func=chi2, k=6)),
        ('model', model)   # ✔ Model added correctly
    ]

    return imbpipeline(steps=final_steps)

def select_model(X, y, pipeline=None):

    classifiers = {}

    c_d1 = {"DummyClassifier": DummyClassifier(strategy='most_frequent')}
    classifiers.update(c_d1)

    c_d4 = {"RandomForestClassifier": RandomForestClassifier()}
    classifiers.update(c_d4)

    c_d5 = {"DecisionTreeClassifier": DecisionTreeClassifier()}
    classifiers.update(c_d5)

    c_d9 = {"KNeighborsClassifier": KNeighborsClassifier()}
    classifiers.update(c_d9)

    c_d10 = {"SVC": SVC(probability=True)}
    classifiers.update(c_d10)

    c_d14 = {
        "MLPClassifier (paper)": MLPClassifier(
            hidden_layer_sizes=(27, 50),
            max_iter=300,
            activation='relu',
            solver='adam',
            random_state=1
        )
    }
    classifiers.update(c_d14)

    cols = ['model', 'run_time', 'roc_auc']
    df_models = pd.DataFrame(columns=cols)

    for key in classifiers:

        start_time = time.time()
        print()
        print("Step 12: model_pipeline run successfully on", key)

        pipeline = model_pipeline(X_train, classifiers[key])
        cv = cross_val_score(pipeline, X, y, cv=10, scoring='roc_auc')

        row = {
            'model': key,
            'run_time': format(round((time.time() - start_time) / 60, 2)),
            'roc_auc': cv.mean()
        }
        df_models = pd.concat([df_models, pd.DataFrame([row])], ignore_index=True)

    df_models = df_models.sort_values(by='roc_auc', ascending=False)
    return df_models

models = select_model(X_train, y_train)
print(models)

selected_model = MLPClassifier(max_iter=300, random_state=1)
bundled_pipeline = model_pipeline(X_train, selected_model)
bundled_pipeline.fit(X_train, y_train)

y_pred = bundled_pipeline.predict(X_test)
print(y_pred)

roc_auc = roc_auc_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('ROC/AUC:', roc_auc)
print('Accuracy:', accuracy)
print('F1 score:', f1)


/tmp/ipython-input-3540541708.py:35: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Weekend'] = df['Weekend'].replace((True, False), (1, 0))
/tmp/ipython-input-3540541708.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Revenue'] = df['Revenue'].replace((True, False), (1, 0))



Step 12: model_pipeline run successfully on DummyClassifier


/tmp/ipython-input-3540541708.py:128: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_models = pd.concat([df_models, pd.DataFrame([row])], ignore_index=True)



Step 12: model_pipeline run successfully on RandomForestClassifier

Step 12: model_pipeline run successfully on DecisionTreeClassifier

Step 12: model_pipeline run successfully on KNeighborsClassifier

Step 12: model_pipeline run successfully on SVC

Step 12: model_pipeline run successfully on MLPClassifier (paper)
                    model run_time   roc_auc
5   MLPClassifier (paper)      1.7  0.903247
4                     SVC      5.6  0.889927
1  RandomForestClassifier     0.49  0.886241
3    KNeighborsClassifier     0.01  0.841013
2  DecisionTreeClassifier     0.02  0.734418
0         DummyClassifier     0.01  0.500000
[0 0 0 ... 0 0 0]
ROC/AUC: 0.8394631573117425
Accuracy: 0.874831035414977
F1 score: 0.6786953504510757


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
